In [9]:
import pandas as pd
import numpy as np
import matplotlib as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report


In [10]:
train = pd.read_csv("adult_train.csv")
test = pd.read_csv("adult_test.csv")

train.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


#### Cleaning missing values

In [11]:
train.replace("?", np.nan, inplace=True)
test.replace("?", np.nan, inplace=True)

train.dropna(inplace=True)
test.dropna(inplace=True)


#### Encode target

In [12]:
train["income"] = train["income"].str.strip().map({"<=50K": 0, ">50K":1})
test["income"] = test["income"].str.strip().map({"<=50K": 0, ">50K":1})

#### Encoding categorical features

In [13]:
cat_cols = ["workclass", "education", "marital-status","occupation","relationship", "race", "sex", "native-country"]

le = LabelEncoder()
for col in cat_cols:
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])
    

#### Split X and y

In [14]:
X_train = train.drop("income", axis=1)
X_test = test.drop("income", axis=1)

y_train = train["income"]
y_test = test["income"]

#### Scale numerical features

In [15]:
num_cols = ["age", "fnlwgt","education-num", "capital-gain", "capital-loss", "hours-per-week"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])


#### Train and Evaluation (Logistic Regression, Naive Bayes, K-Nearest Neighbors, Decision Tree, Support Vector Machine, Random Forest)

In [16]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "Decision tree (C4.5)": DecisionTreeClassifier(criterion="entropy", random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)

}


results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]

    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred),4),
        "F1 score": round(f1_score(y_test, y_pred),4),
        "AUC": round(roc_auc_score(y_test, y_proba),4)
    })


results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
print(results_df.to_string(index=False))

               Model  Accuracy  F1 score    AUC
       Random Forest    0.8508    0.6713 0.9034
                 KNN    0.8213    0.6144 0.8448
 Logistic Regression    0.8204    0.5560 0.8483
                 SVM    0.8093    0.4244 0.8527
Decision tree (C4.5)    0.8089    0.6134 0.7442
         Naive Bayes    0.7990    0.4543 0.8503
